In [31]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [32]:
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('top news in india today')

In [33]:
results

"India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and Bollywood. Hindustan Times: Stay updated with Hindustan Times for the top India news, world events, and breaking stories. Get real-time updates on politics, wars, live cricket scores, entertainment news, and ... Stay updated with the latest breaking news, live updates, photos, videos and top trending stories from India and around the world across politics, economy, sports and more on The Economic Times. Live News: Check our live blog for real-time updates on trending news, breaking news from India, World, Sports, entertainment, politics and more. The Times of India Covers all latest breaking news ... Breaking news from India and the World. Top news stories and videos on Politics, Business, Entertainment, Technology, Sports, Health."

In [34]:
llm = ChatGroq(model="openai/gpt-oss-120b")
llm.invoke('Find me top news in india')

AIMessage(content='**I’m not able to pull live news feeds, so I can’t give you the exact headlines that are trending right this moment.**  \nHowever, I can point you to the best ways to get up‑to‑date Indian news quickly, and I’ll also give you a snapshot of the kinds of stories that have been dominating the Indian news cycle over the past few weeks (based on publicly available reports up to early\u202f2026).\n\n---\n\n## 1. How to Get the **latest** Indian news right now\n\n| Method | How to use it | What you’ll see |\n|--------|---------------|-----------------|\n| **Google News (India)** | Go to <https://news.google.com/topstories?hl=en-IN&gl=IN&ceid=IN:en> | A continuously refreshed “Top Stories” list, grouped by topic (Politics, Business, Sports, etc.). |\n| **News aggregators / apps** | • **Inshorts** (app/website) – 60‑word summaries <br>• **Dailyhunt** – regional language coverage <br>• **Microsoft Start** – personalized feed | Short, mobile‑friendly headlines plus a link to th

In [35]:
@tool
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city
    """
    url = f'https://api.weatherstack.com/current?access_key=a6a3fda9a18275d19df38dcc65e3c293&query={city}'
    response = requests.get(url)
    # Note: Returning stringified JSON is generally safer for tools than raw dicts
    return str(response.json())

In [36]:
tools = [search_tool, get_weather_data]

In [37]:
# 3. Create the LangGraph Agent (Replaces AgentExecutor and Hub Prompt)
agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    # You can optionally pass debug=True here if you want verbose-like console output
)

/tmp/ipykernel_11336/2584645290.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [38]:
# 4. Invoke the Agent (Updated Input Format)
query = "Find the capital of Madhya Pradesh, then find it's current weather condition"

In [39]:
response = agent_executor.invoke({"messages": [("user", query)]})

In [40]:
final_answer = response["messages"][-1].content
print(final_answer)

**Capital of Madhya Pradesh:** **Bhopal**

**Current weather in Bhopal (as of 2026‑09‑09 01:06 local time, Asia/Kolkata):**

| Parameter | Value |
|-----------|-------|
| **Temperature** | 25 °C |
| **Weather condition** | Mist |
| **Feels like** | 29 °C |
| **Humidity** | 95 % |
| **Wind** | 11 km/h from the West (265°) |
| **Pressure** | 1009 hPa |
| **Precipitation** | 0.3 mm |
| **Cloud cover** | 74 % |
| **Visibility** | 2 km |
| **UV index** | 0 (night) |
| **Air‑quality index** (US EPA) | 2 (moderate) |
| **Sunrise / Sunset** | 06:05 AM / 06:30 PM |

So, the capital of Madhya Pradesh is **Bhopal**, and right now it’s a misty night with a mild temperature of about **25 °C** and high humidity.
